# Data Ingest

In [1]:
from pathlib import Path
import re
from collections import defaultdict

import numpy as np
import pandas as pd

EXP = "debug"
N_LEVELS = 4
RIGID = "Rigid"
AFFINE = "Affine"
SYN = "SyN"
STAGES = [RIGID, AFFINE, SYN]

## $P_{min}$

In [2]:
logs = Path("logs", "pmin").glob("*.log")
experiments = dict()
for log_path in logs:
    subject_id = log_path.stem
    experiments[subject_id] = log_path

In [3]:
p_lvl_header = r"DIAGNOSTIC,Iteration,metricValue,convergenceValue,ITERATION_TIME_INDEX,SINCE_LAST.*|  Elapsed time"

dfs = list()
for subject_id, log_path in experiments.items():
    log = log_path.read_text()
    raw_data = [
        [y.strip() for y in x.split("\n") if y.strip() not in ["", "XX"]]
        for i, x in enumerate(re.split(p_lvl_header, log))
        if i % 5 != 0
    ]
    for stage in STAGES:
        for level in range(N_LEVELS):
            table = defaultdict(list)
            for line in raw_data[level + N_LEVELS * STAGES.index(stage)]:
                if not (
                    line.startswith("2DIAGNOSTIC") or line.startswith("1DIAGNOSTIC")
                ):
                    # print(f"Skipping line: {line}")
                    continue

                cols = line.split(",")
                table["iteration"].append(int(cols[1]))
                table["metricValue"].append(np.float64(cols[2]))

                if line.startswith("2DIAGNOSTIC"):  # Rigid + Affine
                    table["lipschitz"].append(np.float64(cols[6]))
                    table["params_2norm"].append(np.float64(cols[7]))
                    table["grad_2norm"].append(np.float64(cols[8]))

                elif line.startswith("1DIAGNOSTIC"):  # SyN
                    # Moving
                    table["moving_lipschitz"].append(np.float64(cols[6]))
                    table["moving_params_2norm"].append(np.float64(cols[7]))
                    table["moving_grad_2norm"].append(np.float64(cols[8]))
                    # Fixed
                    table["fixed_lipschitz"].append(np.float64(cols[9]))
                    table["fixed_params_2norm"].append(np.float64(cols[10]))
                    table["fixed_grad_2norm"].append(np.float64(cols[11]))

            common_data = {
                "exp": EXP,
                "stage": stage,
                "level": level,
                "subject_id": subject_id,
                "iteration": table["iteration"],
                "metricValue": table["metricValue"],
            }

            if line.startswith("2DIAGNOSTIC"):  # Rigid + Affine
                dfs.append(
                    pd.DataFrame(
                        data=common_data
                        | {
                            "lipschitz": table["lipschitz"],
                            "params_2norm": table["params_2norm"],
                            "grad_2norm": table["grad_2norm"],
                        }
                    )
                )
            elif line.startswith("1DIAGNOSTIC"):  # SyN
                # Moving
                dfs.append(
                    pd.DataFrame(
                        data=common_data
                        | {
                            "direction": "moving",
                            "lipschitz": table["moving_lipschitz"],
                            "params_2norm": table["moving_params_2norm"],
                            "grad_2norm": table["moving_grad_2norm"],
                        }
                    )
                )
                # Fixed
                dfs.append(
                    pd.DataFrame(
                        data=common_data
                        | {
                            "direction": "fixed",
                            "lipschitz": table["fixed_lipschitz"],
                            "params_2norm": table["fixed_params_2norm"],
                            "grad_2norm": table["fixed_grad_2norm"],
                        }
                    )
                )

pmin_df = pd.concat(dfs, ignore_index=True)
pmin_df = pmin_df.astype(
    {
        "exp": str,
        "stage": str,
        "level": int,
        "iteration": int,
        "subject_id": str,
        "direction": str,
        "lipschitz": np.float64,
        "params_2norm": np.float64,
        "grad_2norm": np.float64,
        "metricValue": np.float64,
    }
)
pmin_df["pmin"] = np.log2(
    (4 + 3 * np.sqrt(2))
    / pmin_df["grad_2norm"]
    * pmin_df["lipschitz"]
    * pmin_df["params_2norm"]
)

## VPREC

In [4]:
PLOT_BOUNDARIES = False

In [5]:
from pathlib import Path
import re

DEBUG = True
N_SUBJECT = 50

FILENAME_PATTERN = re.compile(
    r"(?P<subject_id>\d+)-r(?P<range>\d+)-p(?P<precision>\d+)\.log"
)

figures = Path("paper", "figures")
figures.mkdir(exist_ok=True, parents=True)

In [6]:
from typing import Iterable


def get_failed_exec(logs: Path | Iterable[Path]) -> int:
    if isinstance(logs, Path):
        logs = [logs]

    failed = list()
    for log in logs:
        if log.read_text().count("Elapsed time (stage 0):") != 4:
            failed.append(log)
    return failed


In [7]:
# Group logs
import re
from collections import defaultdict


logs = list(Path("logs", "vprec-space_search").rglob("*.log"))

experiements = defaultdict(list)
for log in logs:
    if "binary64" in log.name:
        exp_id = f"{log.parent.name}-binary64"
    else:
        m = FILENAME_PATTERN.match(log.name)
        if not m:
            print(log)
            raise ValueError("Invalid log file name")
        exp_id = f"{log.parent.name}-r{m.group('range')}-p{m.group('precision')}"

    experiements[exp_id].append(log)


In [8]:
# Count failed execution for each range-precision pair
n_failed = defaultdict(dict)
for task_id, logs in experiements.items():
    failed = get_failed_exec(logs)

    if "binary64" in task_id:
        continue
    range_ = int(FILENAME_PATTERN.match(logs[0].name).group("range"))
    precision_ = int(FILENAME_PATTERN.match(logs[0].name).group("precision"))
    n_failed[range_][precision_] = len(failed)

    if len(failed) > 0:
        print(f"Task {task_id} (r{range_}-p{precision_}): {len(failed)} failed")
    if DEBUG and len(failed) != len(logs):
        for log in failed:
            print(log, end="\n\n")

if PLOT_BOUNDARIES:
    # Manually add failure for range 6
    n_failed[6] = {r: N_SUBJECT for r in range(7, 24)}
else:
    del n_failed[8][6]
    del n_failed[7][6]


Task 0011-r7-p6 (r7-p6): 50 failed
Task 0011-r8-p6 (r8-p6): 50 failed
Task 0100-r7-p6 (r7-p6): 50 failed
Task 0100-r8-p6 (r8-p6): 50 failed
Task 0111-r7-p6 (r7-p6): 50 failed
Task 0111-r8-p6 (r8-p6): 50 failed
Task 0110-r7-p6 (r7-p6): 50 failed
Task 0110-r8-p6 (r8-p6): 50 failed
Task 1111-r7-p6 (r7-p6): 50 failed
Task 1111-r8-p6 (r8-p6): 50 failed
Task 0001-r7-p6 (r7-p6): 50 failed
Task 0001-r8-p6 (r8-p6): 50 failed


In [9]:
import pandas as pd
import numpy as np


p_lvl_header = r"DIAGNOSTIC,Iteration,metricValue,convergenceValue,ITERATION_TIME_INDEX,SINCE_LAST|  Elapsed time"

dfs = list()

for exp_id, logs in experiements.items():
    for filename in logs:
        # Extract subject_id from log
        subject_id = filename.stem.split("-")[0]

        # Filter out the header from the stages
        txt = filename.read_text()
        all_data = [
            x
            for x in re.split(p_lvl_header, txt)[1:]
            if not x.startswith(" (stage 0):")
        ]

        # 3 stages with 4 levels of resolution each
        for stage_id, stage in enumerate(STAGES, 1):
            for i, level in enumerate(all_data[4 * (stage_id - 1) : 4 * stage_id]):
                table = defaultdict(list)
                for row in re.split(r"\n", level.strip("XX").strip()):
                    # Skip invalid rows
                    # e.g. error messages or write volumes to disk
                    if not ("2DIAGNOSTIC" in row or "1DIAGNOSTIC" in row):
                        continue

                    cols = row.split(",")
                    table["iteration"].append(int(cols[1]))
                    table["metric"].append(np.float64(cols[2]))
                    table["convergence_value"].append(np.float64(cols[3]))
                    table["total_time"].append(np.float64(cols[4]))
                    table["since_last"].append(np.float64(cols[5]))

                data_format = "-".join(exp_id.split("-")[1:])
                if data_format == "binary64":
                    precision = 53
                    range_ = 11
                else:
                    precision = int(data_format.split("-")[1].removeprefix("p"))
                    range_ = int(data_format.split("-")[0].removeprefix("r"))

                dfs.append(
                    pd.DataFrame(
                        data={
                            "subject": subject_id,
                            "exp_id": exp_id.split("-")[0],
                            "data_format": data_format,
                            "precision": precision,
                            "range": range_,
                            "stage": stage,
                            "level": i,
                            "iteration": table["iteration"],
                            "metric": table["metric"],
                            "convergence_value": table["convergence_value"],
                            "total_time": table["total_time"],
                            "since_last": table["since_last"],
                        }
                    )
                )

vprec_df = pd.concat(dfs, ignore_index=True)
vprec_df = vprec_df.astype(
    {
        "stage": str,
        "level": int,
        "iteration": int,
        "metric": np.float64,
        "convergence_value": np.float64,
        "total_time": np.float64,
        "since_last": np.float64,
    }
)


# Step 1: extract the Binary64 metric per group
vprec_df
ref = (
    vprec_df[vprec_df["data_format"] == "binary64"]
    .groupby(["subject", "stage", "level", "iteration"])["metric"]
    .mean()  # or .first() if unique
    .rename("binary64_metric")
)

# Step 2: join back to the original DataFrame
vprec_df = vprec_df.merge(
    ref, on=["subject", "stage", "level", "iteration"], how="left"
)

# Step 3: compute the difference
vprec_df["metric_rel_diff"] = (
    vprec_df["metric"] - vprec_df["binary64_metric"]
) / np.abs(vprec_df["binary64_metric"])

## Merging $P_{min}$ and VPREC to compute diff $p_{min}$ at each iteration

In [10]:
TRESHOLD = 1e-3

In [11]:
# VPREC minimum precision per stage, level, iteration
vprec_means = (
    vprec_df.groupby(["exp_id", "precision", "range", "stage", "level", "iteration"])[
        "metric_rel_diff"
    ]
    .mean()
    .rename("mean_rel_diff")
    .reset_index()
)
vprec_means = vprec_means[vprec_means["mean_rel_diff"] <= TRESHOLD]

vprec_pmin = (
    vprec_means.groupby(["exp_id", "stage", "level", "iteration"])["precision"]
    .min()
    .rename("vprec_pmin")
    .reset_index()
)

In [12]:
# PMIN mean estimate per stage, level, iteration, direction
pmin_means = (
    pmin_df.groupby(["stage", "level", "iteration", "direction"])["pmin"]
    .mean()
    .rename("estimated_pmin")
    .reset_index()
)

In [13]:
plot_data = vprec_pmin.merge(pmin_means, on=["stage", "level", "iteration"], how="left")
plot_data["diff_pmin"] = plot_data["vprec_pmin"] - plot_data["estimated_pmin"]
plot_data

,exp_id,stage,level,iteration,vprec_pmin,direction,estimated_pmin,diff_pmin
0,0000,Affine,0,1,53,nan,8.228812,44.771188
1,0000,Affine,0,2,53,nan,7.628706,45.371294
2,0000,Affine,0,3,53,nan,7.100903,45.899097
3,0000,Affine,0,4,53,nan,6.695640,46.304360
4,0000,Affine,0,5,53,nan,6.375329,46.624671
...,...,...,...,...,...,...,...,...
6422,1111,SyN,3,18,11,moving,12.554447,-1.554447
6423,1111,SyN,3,19,11,fixed,12.620620,-1.620620
6424,1111,SyN,3,19,11,moving,12.604794,-1.604794
6425,1111,SyN,3,20,11,fixed,12.587984,-1.587984


# Plotting

In [14]:
import plotly.graph_objects as go
from plotly.colors import hex_to_rgb


def hex_to_rgba(hex_color: str, alpha: float) -> str:
    r, g, b = hex_to_rgb(hex_color)
    return f"rgba({r}, {g}, {b}, {alpha})"


def plot_values(
    fig: go.Figure,
    df: pd.DataFrame,
    metric: list[str] | str,
    *,
    row: int,
    col: int,
    name: str,
    color: str,
    dash: list[str] | str = "solid",
    showlegend: bool = False,
):
    for m, d in zip(metric, dash):
        iterations = sorted(df["iteration"].unique())
        mean_values = df.groupby("iteration")[m].mean()

        # Mean line
        fig.add_trace(
            go.Scatter(
                x=iterations,
                y=mean_values,
                mode="lines",
                line=dict(color=color, width=3, dash=d),
                name=name,
                showlegend=showlegend,
            ),
            row=row,
            col=col,
        )


In [15]:
from typing import Callable
import plotly.graph_objects as go
from plotly.subplots import make_subplots

STAGE_to_EXP = {
    "Rigid": "0111",
    "Affine": "0011",
    "SyN": "0001",
}


def viz_pmin(
    df: pd.DataFrame,
    *,
    boundary_func: Callable,
    metric: list[str] | str,
    dash: list[str] | str = "solid",
    title: str,
    out_file: Path | None = None,
):
    BOKEH_COLORBLIND = (
        "#0072B2",  # 0
        "#E69F00",  # 1
        "#F0E442",  # 2
        "#009E73",  # 3
        "#56B4E9",  # 4
        "#D55E00",  # 5
        "#CC79A7",  # 6
        "#000000",  # 7
    )
    COLORS = {
        "Rigid": BOKEH_COLORBLIND[3],
        "Affine": BOKEH_COLORBLIND[6],
        "SyN (Moving)": BOKEH_COLORBLIND[0],
        "SyN (Fixed)": BOKEH_COLORBLIND[1],
    }

    def add_hline(fig, y, text):
        # Only show the horizontal line in the last column
        fig.add_hline(
            y,
            line=dict(dash="dot"),
        )
        fig.add_hline(
            y,
            col=N_LEVELS,
            line=dict(dash="dot"),
            annotation_text=text,
            annotation_position="right",
            annotation_xref="paper",
            annotation_font_size=12,
            annotation_showarrow=False,
            annotation_font_color="black",
        )

    metric = [metric] if isinstance(metric, str) else metric
    dash = [dash] if isinstance(dash, str) else dash
    assert len(metric) == len(dash)

    df_ = df.copy()
    # for subject_id in df_["subject_id"].unique():
    #     df_ = df_[df_["subject_id"] == subject_id]

    # Get sorted unique STAGES and levels
    levels = sorted(df_["level"].unique())

    # Create subplot figure
    fig = make_subplots(
        rows=len(STAGES),
        cols=N_LEVELS,
        subplot_titles=[
            f"[{STAGE_to_EXP[stage]}] {stage}: Level {level}"
            for stage in STAGES
            for level in levels
        ],
        shared_xaxes=False,
        shared_yaxes=True,
        horizontal_spacing=0.025,
        vertical_spacing=0.1,
    )
    # Update subplot titles font size
    for annotation in fig.layout.annotations:
        annotation.font = dict(size=18)

    # Map stage to row index
    stage_to_row = {stage: i + 1 for i, stage in enumerate(STAGES)}
    level_to_col = {level: i + 1 for i, level in enumerate(levels)}

    # Add traces
    for i, stage in enumerate(STAGES):
        for j, level in enumerate(levels):
            sub_df = df_[
                (df_["stage"] == stage)
                & (df_["level"] == level)
                & (df_["exp_id"] == STAGE_to_EXP[stage])
            ].copy()
            row = stage_to_row[stage]
            col = level_to_col[level]

            showlegend = False
            if stage in [RIGID, AFFINE]:
                boundary_func(
                    fig,
                    sub_df,
                    row=row,
                    col=col,
                    metric=metric,
                    showlegend=showlegend,
                    name=stage,
                    color=COLORS[stage],
                    dash=dash,
                )
            elif stage == SYN:
                moving_df = sub_df[sub_df["direction"] == "moving"]
                fixed_df = sub_df[sub_df["direction"] == "fixed"]

                # Moving
                boundary_func(
                    fig,
                    moving_df,
                    row=row,
                    col=col,
                    metric=metric,
                    showlegend=showlegend,
                    name="SyN (Moving)",
                    color=COLORS["SyN (Moving)"],
                    dash=dash,
                )

                # Fixed
                boundary_func(
                    fig,
                    fixed_df,
                    row=row,
                    col=col,
                    metric=metric,
                    showlegend=showlegend,
                    name="SyN (Fixed)",
                    color=COLORS["SyN (Fixed)"],
                    dash=dash,
                )

    # Legend items
    for m, d in zip(metric, dash):
        fig.add_trace(
            go.Scatter(
                x=[1],
                y=[0],
                mode="lines",
                line=dict(dash=d, color="black"),
                name=m,
                showlegend=True,
                hoverinfo="skip",
            ),
            row=1,
            col=1,
        )

    # Only add this ONCE (e.g., at subplot [1,1] or outside the loop)
    fig.add_trace(
        go.Scatter(
            x=[1],
            y=[0],
            mode="lines",
            line=dict(dash="dot", color="black"),
            name="hardware support",
            showlegend=True,
            hoverinfo="skip",
        ),
        row=1,
        col=1,
    )
    # add_hline(fig, y=53, text="FP64")
    add_hline(fig, y=24, text="FP32")
    add_hline(fig, y=16, text="FP24")
    add_hline(fig, y=11, text="TF32")
    add_hline(fig, y=8, text="BF16")

    # Layout adjustments
    fig.update_layout(
        height=len(STAGES) * 400,
        width=1200,
        font=dict(
            size=24,
            color="black",
        ),
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.025,  # position above the subplots
            xanchor="left",
            x=0,
            traceorder="normal",
        ),
        title=title,
        margin=dict(l=10, r=40, t=150, b=10),
        plot_bgcolor="white",
    )
    for i in range(1, len(STAGES) * N_LEVELS + 1):
        fig.update_layout(
            {
                f"xaxis{i}": dict(showline=True, linewidth=2, linecolor="black"),
                f"yaxis{i}": dict(showline=True, linewidth=2, linecolor="black"),
            }
        )

    # Set x-axis and y-axis titles
    for i, _ in enumerate(levels):
        fig.update_xaxes(title_text="nth Iteration", row=len(STAGES), col=i + 1)

    fig.show()
    # Save the figure
    if out_file is not None:
        fig.update_layout(
            title=None,
            margin=dict(l=10, r=40, t=10, b=10),
        )
        fig.write_image(
            out_file,
            width=1200,
            height=len(STAGES) * 400,
            scale=2,
        )


In [16]:
fig_dir = Path("paper", "figures")

viz_pmin(
    plot_data,
    metric=["vprec_pmin", "estimated_pmin"],
    dash=["solid", "dash"],
    boundary_func=plot_values,
    title="pmin at each iteration",
    out_file=fig_dir / "pmin-overview.pdf",
)
